In [ ]:
# pyright: reportGeneralTypeIssues=false, reportUnknownMemberType=false, reportUnknownVariableType=false, reportUnknownArgumentType=false
# ruff: noqa
# pylint: skip-file

# NHANES Diabetes Prediction - Bayesian Hyperparameter Optimization

This notebook implements Bayesian optimization using Optuna with W&B tracking for the LightGBM diabetes screening model.

**Optimization target**: Recall (prioritizing detection of positive cases for screening purposes)

## 1. Setup and Configuration

In [ ]:
import lightgbm as lgb
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import optuna
import pandas as pd
import wandb
from optuna.integration.wandb import WeightsAndBiasesCallback
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    confusion_matrix,
    recall_score,
    precision_score,
    f1_score,
    precision_recall_curve,
    auc,
    average_precision_score,
    brier_score_loss,
)
from sklearn.model_selection import train_test_split  # pyright: ignore[reportUnknownVariableType]
from sklearn.model_selection import StratifiedKFold
from sklearn.utils import resample

import numpy as np
import shap

In [ ]:
# Configuration

# Weights and Biases
WANDB_PROJECT = "Model exploration for Diabetes Prediction"
ENTITY = "fastegiano-tesis"

# Optuna
STORAGE = "sqlite:///optuna_diabetes.db"
STUDY_NAME = "exp7_metabolic_history_v3_PA_zeros"
PROJECT_NAME = "Model exploration for Diabetes Prediction"
RANDOM_STATE = 37
N_TRIALS = 200

# Post-training options
USE_CALIBRATION = False  # Toggle probability calibration on/off

# Categorical features
CAT_FEATURES = [
    "education_level",
    "has_partner",
    "had_partner",
    "is_female",
    "ever_smoker",
    "is_current_smoker",
    "drinking_frequency",
    "told_high_bp",           # NEW — binary: ever told high blood pressure
    "told_high_cholesterol",  # NEW — binary: ever told high cholesterol
]

## 2. Data Loading and Preparation

In [ ]:
data = pd.read_csv("../../../dataset/output/processed_data_combined_metabolic_history_v2.csv")  # pyright: ignore[reportUnknownMemberType]
data.head()

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
# NOTE: survey_weight is dropped because this model is trained as a predictive
# screening tool on the observed sample, not for population-level prevalence
# estimation. This decision should be documented in the thesis methodology section.
WEIGHTS = data.pop("survey_weight")  # pyright: ignore[reportUnknownMemberType]
#data.drop(columns="survey_weight", inplace=True, errors="ignore")
print(f"Data shape: {data.shape}")

### Feature Engineering

In [ ]:
data["waist_to_height_ratio"] = data["BMXWAIST"] / data["BMXHT"]
data["age_bmi_interaction"] = data["RIDAGEYR"] * data["BMXBMI"]

In [ ]:
def create_age_bins(
    df: pd.DataFrame,
    age_column: str = "RIDAGEYR",
    age_bins: tuple[int, ...] = (18, 45, 65, 79),
    age_labels: tuple[str, ...] = ("young_adult", "middle_age", "senior"),
    elderly_label: str = "elderly",
    unknown_label: str = "age_unknown",
    elderly_top_coded_age: int = 80,
) -> pd.DataFrame:
    """Bin age into categories and one-hot encode."""
    age = df[age_column].copy()
    age_group = pd.Series(index=df.index, dtype="object")

    missing_mask = age.isna()
    elderly_mask = age >= elderly_top_coded_age

    age_group[elderly_mask] = elderly_label

    valid_mask = ~missing_mask & ~elderly_mask
    age_group[valid_mask] = pd.cut(
        age[valid_mask],
        bins=list(age_bins),
        labels=age_labels,
        right=False,
    )

    age_group[missing_mask] = unknown_label

    dummies = pd.get_dummies(age_group, prefix="age", dtype=int)

    unknown_col = f"age_{unknown_label}"
    if unknown_col in dummies.columns and missing_mask.sum() == 0:
        dummies = dummies.drop(columns=[unknown_col])

    return pd.concat([df, dummies], axis=1)

In [ ]:
data = create_age_bins(data)

### X y Split

In [ ]:
TARGET_COL = "has_diabetes_or_prediabetes"

# All target/meta columns that must NOT be in features
DROP_FROM_FEATURES = [
    TARGET_COL,
    "cycle",
    "lab_positive",     # Lab-defined target — NOT a feature (requires blood draw)
    "undiagnosed",      # Derived target — NOT a feature
]

# Drop only columns that exist (defensive, in case of column name changes)
cols_to_drop = [c for c in DROP_FROM_FEATURES if c in data.columns]
X = data.drop(columns=cols_to_drop)
y = data[TARGET_COL]

In [ ]:
# First split: separate out the untouchable test set (10%)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.10, stratify=y, random_state=RANDOM_STATE
)

# Second split: separate early-stopping validation from threshold-tuning validation
# We split the remaining 90% into ~78% train, ~11% val_es, ~11% val_thresh
X_train_full, X_val_thresh, y_train_full, y_val_thresh = train_test_split(
    X_trainval,
    y_trainval,
    test_size=0.12,
    stratify=y_trainval,
    random_state=RANDOM_STATE,
)

X_train, X_val_es, y_train, y_val_es = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.125,  # 0.125 of 78% ≈ 10% of total
    stratify=y_train_full,
    random_state=RANDOM_STATE,
)

print(f"Train: {len(X_train)} | Val ES: {len(X_val_es)} | Val Thresh: {len(X_val_thresh)} | Test: {len(X_test)}")
print(f"Prevalence — Train: {y_train.mean():.1%} | Val ES: {y_val_es.mean():.1%} | Val Thresh: {y_val_thresh.mean():.1%} | Test: {y_test.mean():.1%}")

### Null revision

In [ ]:
# Check for null values in all splits
print("=" * 70)
print("NULL VALUES SUMMARY")
print("=" * 70)

null_summary = pd.DataFrame({
    "X_train_null_%": (X_train.isnull().sum() / len(X_train)) * 100,
    "X_val_es_null_%": (X_val_es.isnull().sum() / len(X_val_es)) * 100,
    "X_val_thresh_null_%": (X_val_thresh.isnull().sum() / len(X_val_thresh)) * 100,
    "X_test_null_%": (X_test.isnull().sum() / len(X_test)) * 100,
})

null_summary = null_summary[
    (null_summary > 0).any(axis=1)
]
null_summary = null_summary.sort_values("X_train_null_%", ascending=False)

print(f"\nX_train shape: {X_train.shape}")
print(f"X_val_es shape: {X_val_es.shape}")
print(f"X_val_thresh shape: {X_val_thresh.shape}")
print(f"X_test shape: {X_test.shape}")
print(
    f"\nTotal nulls — Train: {X_train.isnull().sum().sum()}, "
    f"Val ES: {X_val_es.isnull().sum().sum()}, "
    f"Val Thresh: {X_val_thresh.isnull().sum().sum()}, "
    f"Test: {X_test.isnull().sum().sum()}"
)

if len(null_summary) > 0:
    print(f"\nColumns with null values:\n")
    print(null_summary.to_string())
else:
    print("\nNo null values found in any split!")

print("\n" + "=" * 70)

---

## 3. Model preparation

## Weights & Biases initialization

In [ ]:
WANDB_KWARGS = {  # type: ignore
    "entity": ENTITY,
    "project": WANDB_PROJECT,
    "name": STUDY_NAME,
    "tags": ["lightgbm", "metabolic-history", "self-report-target"],
    "notes": "Added told_high_bp, told_high_cholesterol. Self-report target. 200 trials.",
    "config": {
        "random_state": RANDOM_STATE,
        "n_trials": N_TRIALS,
        "metric": "average_precision",
    },
}

## Optimization set up

In [ ]:
from typing import Any

# Compute scale_pos_weight from training set class distribution
_neg_count = (y_train == 0).sum()
_pos_count = (y_train == 1).sum()
_default_scale_pos_weight = _neg_count / _pos_count
print(f"Class ratio (neg/pos): {_default_scale_pos_weight:.2f}")

fixed_params: dict[str, Any] = {
    "objective": "binary",
    "metric": "binary_logloss",
    "verbosity": -1,
    "n_estimators": 2000,
    "bagging_seed": RANDOM_STATE,
    "feature_fraction_seed": RANDOM_STATE,
    "random_state": RANDOM_STATE,
}

In [ ]:
def objective(trial, fixed_params=fixed_params):  # type: ignore

    max_depth = trial.suggest_int("max_depth", 3, 8)  # type: ignore

    params = {  # pyright: ignore[reportUnknownVariableType]
        **fixed_params,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),  # type: ignore
        "max_depth": max_depth,  # type: ignore
        "num_leaves": trial.suggest_int("num_leaves", 2, 2**max_depth - 1),  # type: ignore
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 5, 100),  # type: ignore
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.1, 1.0),  # type: ignore
        "bagging_freq": trial.suggest_int("bagging_freq", 3, 10),  # type: ignore
        "feature_fraction": trial.suggest_float("feature_fraction", 0.1, 1.0),  # type: ignore
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, _default_scale_pos_weight * 1.5),  # type: ignore
    }

    kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
    val_scores = []
    train_scores = []
    n_iterations = []

    for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(X_train, y_train)):  # type: ignore
        X_cv_tr, X_cv_val = X_train.iloc[train_idx], X_train.iloc[val_idx]  # type: ignore
        y_cv_tr, y_cv_val = y_train.iloc[train_idx], y_train.iloc[val_idx]  # type: ignore

        model = lgb.LGBMClassifier(**params)  # type: ignore
        model.fit(  # type: ignore
            X_cv_tr,
            y_cv_tr,  # type: ignore
            eval_set=[(X_cv_val, y_cv_val)],
            callbacks=[lgb.early_stopping(50, verbose=False)],
            categorical_feature=CAT_FEATURES,
        )

        n_iterations.append(model.best_iteration_)

        y_cv_val_proba = model.predict_proba(X_cv_val)[:, 1]
        y_cv_tr_proba = model.predict_proba(X_cv_tr)[:, 1]

        val_ap = average_precision_score(y_cv_val, y_cv_val_proba)
        train_ap = average_precision_score(y_cv_tr, y_cv_tr_proba)

        val_scores.append(val_ap)
        train_scores.append(train_ap)

    mean_val_ap = np.mean(val_scores)
    mean_train_ap = np.mean(train_scores)
    overfitting_gap = mean_train_ap - mean_val_ap

    wandb.log(
        {
            "mean_val_ap": mean_val_ap,
            "std_val_ap": np.std(val_scores),
            "min_fold_ap": np.min(val_scores),
            "overfitting_gap": overfitting_gap,
            "mean_n_iterations": np.mean(n_iterations),
        },
        step=trial.number,
    )

    return mean_val_ap

## 4. Run Optimization

In [ ]:
# Delete all studies
"""
optuna.delete_study(
    study_name=STUDY_NAME,
    storage="sqlite:///optuna.db"  # change if your storage is different
)
"""

In [ ]:
study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE,
    load_if_exists=True,
    direction="maximize",
)

In [ ]:
wandb_callback = WeightsAndBiasesCallback(
    metric_name="average_precision",
    wandb_kwargs=WANDB_KWARGS,  # type: ignore
)

study.optimize(objective, n_trials=N_TRIALS, callbacks=[wandb_callback])  # type: ignore

wandb.finish()

## 5. Results Analysis

In [ ]:
# Display best params from the Optuna study
best_params = study.best_params
print("Optimized parameters:")
for k, v in best_params.items():
    print(f"{k}: {v}")

print(f"\nBest AP (study.best_value): {study.best_value:.4f}")

## 6. Best Model Evaluation

In [ ]:
from typing import Any


def get_full_params(
    fixed_params: dict[str, Any], best_params: dict[str, Any]
) -> dict[str, Any]:
    return {**fixed_params, **best_params}

In [ ]:
best_model = lgb.LGBMClassifier(**get_full_params(fixed_params, study.best_params))

best_model.fit(
    X_train,
    y_train,
    eval_set=[(X_train, y_train), (X_val_es, y_val_es)],
    eval_names=["train", "valid"],
    callbacks=[lgb.early_stopping(50, verbose=False)],
    categorical_feature=CAT_FEATURES,
)  # type: ignore

y_pred = best_model.predict(X_test)  # type: ignore

### Probability Calibration (Optional)

Controlled by `USE_CALIBRATION` flag. When enabled, calibrates probability estimates using isotonic regression on the validation set.

In [ ]:
# Conditionally calibrate and select inference model
if USE_CALIBRATION:
    calibrated_model = CalibratedClassifierCV(
        best_model, method="isotonic", cv="prefit"
    )
    calibrated_model.fit(X_val_thresh, y_val_thresh)
    inference_model = calibrated_model
    print("Using CALIBRATED model (isotonic regression on val_thresh set)")
else:
    inference_model = best_model
    print("Using UNCALIBRATED model (raw probabilities)")

In [ ]:
recall_test_positive = np.round(recall_score(y_test, y_pred, pos_label=1), 2)  # type: ignore
recall_test_negative = np.round(recall_score(y_test, y_pred, pos_label=0), 2)  # type: ignore

In [ ]:
print(f"y_test value counts:\n{y_test.value_counts()}")
print(f"y_pred value counts:\n{pd.Series(y_pred).value_counts()}")
print(f"Recall test (pos_label=1): {recall_test_positive}")
print(f"Recall test (pos_label=0): {recall_test_negative}")

### Training Progress - Binary Log Loss

In [ ]:
# Extract evaluation results
results = best_model.evals_result_

# Plot the progression
fig_overfit, ax = plt.subplots(figsize=(10, 6))
ax.plot(results["train"]["binary_logloss"], label="Train", linewidth=2)
ax.plot(results["valid"]["binary_logloss"], label="Valid", linewidth=2)
ax.set_xlabel("Iteration")
ax.set_ylabel("Binary Log Loss")
ax.set_title("Binary Log Loss Over Training Iterations")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)  # type: ignore
fig_cm_05, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm, display_labels=["No Diabetes", "Diabetes"]
)
disp.plot(cmap="Blues", ax=ax)  # type: ignore
ax.set_title(f"Confusion Matrix — Threshold = 0.50\nRecall: {recall_test_positive}")
plt.tight_layout()
plt.show()  # type: ignore

## 7. SHAP Analysis

In [ ]:
# Compute SHAP values using TreeExplainer (exact, fast for LightGBM)
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

# For binary classification, shap_values is a list [class_0, class_1]
# We want the positive class (index 1)
if isinstance(shap_values, list):
    shap_values_pos = shap_values[1]
else:
    shap_values_pos = shap_values

# --- Global importance: mean |SHAP| bar plot ---
fig_shap_global, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values_pos, X_test, plot_type="bar", show=False, max_display=20)
plt.title("Global Feature Importance (mean |SHAP|)")
plt.tight_layout()
fig_shap_global = plt.gcf()
plt.show()

In [ ]:
# --- SHAP beeswarm plot: shows direction + magnitude of feature effects ---
fig_shap_beeswarm, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values_pos, X_test, show=False, max_display=20)
plt.title("SHAP Beeswarm — Feature Impact on Positive Class Prediction")
plt.tight_layout()
fig_shap_beeswarm = plt.gcf()
plt.show()

In [ ]:
# --- Dependence plots for top 3 features ---
mean_abs_shap = np.abs(shap_values_pos).mean(axis=0)
top_features = pd.Series(mean_abs_shap, index=X_test.columns).sort_values(ascending=False).head(3).index.tolist()

fig_shap_dep, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, feat in enumerate(top_features):
    shap.dependence_plot(feat, shap_values_pos, X_test, ax=axes[i], show=False)
plt.suptitle("SHAP Dependence Plots — Top 3 Features", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Build SHAP importance DataFrame for later W&B logging
shap_importance = pd.Series(mean_abs_shap, index=X_test.columns).sort_values(ascending=False)
print("\nSHAP Global Importance (top 10):")
for feat, val in shap_importance.head(10).items():
    print(f"  {feat}: {val:.4f}")

In [ ]:
# Threshold optimization using val_thresh set (NOT used for early stopping)
y_proba = inference_model.predict_proba(X_val_thresh)[:, 1]

# Calculate PR curve
precisions, recalls, thresholds = precision_recall_curve(y_val_thresh, y_proba)
pr_auc = auc(recalls, precisions)

# Create figure with 2 subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: Precision-Recall Curve ---
ax1 = axes[0]
ax1.plot(recalls, precisions, "b-", linewidth=2, label=f"PR Curve (AUC={pr_auc:.3f})")
ax1.fill_between(recalls, precisions, alpha=0.2)

target_recalls = [0.80, 0.75, 0.70, 0.50]
colors = ["red", "orange", "green", "purple"]

for target, color in zip(target_recalls, colors):
    idx = np.where(recalls[:-1] >= target)[0]
    if len(idx) > 0:
        i = idx[-1]
        thresh = thresholds[i]
        ax1.scatter(
            recalls[i],
            precisions[i],
            c=color,
            s=100,
            zorder=5,
            label=f"Recall={target:.0%} (thresh={thresh:.3f}, prec={precisions[i]:.1%})",
        )

baseline = y_val_thresh.mean()
ax1.axhline(
    y=baseline,
    color="gray",
    linestyle="--",
    label=f"Baseline (prevalence={baseline:.1%})",
)

ax1.set_xlabel("Recall (Sensitivity)", fontsize=12)
ax1.set_ylabel("Precision (PPV)", fontsize=12)
calibration_label = "Calibrated" if USE_CALIBRATION else "Uncalibrated"
ax1.set_title(f"Precision-Recall Curve ({calibration_label})", fontsize=14)
ax1.legend(loc="upper right", fontsize=9)
ax1.set_xlim([0, 1.02])
ax1.set_ylim([0, 1.02])
ax1.grid(True, alpha=0.3)

# --- Plot 2: Threshold vs Metrics ---
ax2 = axes[1]

thresh_range = np.linspace(0.05, 0.6, 100)
recall_at_thresh = []
precision_at_thresh = []
f1_at_thresh = []
flagged_pct = []

for t in thresh_range:
    y_pred_t = (y_proba >= t).astype(int)
    r = recall_score(y_val_thresh, y_pred_t, zero_division=0)
    p = precision_score(y_val_thresh, y_pred_t, zero_division=0)
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    recall_at_thresh.append(r)
    precision_at_thresh.append(p)
    f1_at_thresh.append(f1)
    flagged_pct.append(y_pred_t.mean())

ax2.plot(thresh_range, recall_at_thresh, "b-", linewidth=2, label="Recall")
ax2.plot(thresh_range, precision_at_thresh, "r-", linewidth=2, label="Precision")
ax2.plot(thresh_range, f1_at_thresh, "g--", linewidth=2, label="F1 Score")
ax2.plot(thresh_range, flagged_pct, "k:", linewidth=2, label="% Flagged")

ax2.axvline(x=0.5, color="gray", linestyle="--", alpha=0.7, label="Default (0.5)")

target_80_idx = np.argmin(np.abs(np.array(recall_at_thresh) - 0.80))
optimal_thresh = thresh_range[target_80_idx]
ax2.axvline(
    x=optimal_thresh,
    color="red",
    linestyle="--",
    alpha=0.7,
    label=f"80% Recall (thresh={optimal_thresh:.3f})",
)

ax2.set_xlabel("Decision Threshold", fontsize=12)
ax2.set_ylabel("Score", fontsize=12)
ax2.set_title(f"Metrics vs Decision Threshold ({calibration_label})", fontsize=14)
ax2.legend(loc="center right", fontsize=9)
ax2.set_xlim([0.05, 0.6])
ax2.set_ylim([0, 1.02])
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("precision_recall_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Print Summary Table ---
print("\n" + "=" * 70)
print(f"THRESHOLD ANALYSIS SUMMARY ({calibration_label.upper()} PROBABILITIES)")
print("=" * 70)
print(f"{'Threshold':<12} {'Recall':<12} {'Precision':<12} {'F1':<12} {'Flagged':<12}")
print("-" * 70)

for thresh in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]:
    y_pred_t = (y_proba >= thresh).astype(int)
    r = recall_score(y_val_thresh, y_pred_t, zero_division=0)
    p = precision_score(y_val_thresh, y_pred_t, zero_division=0)
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    n_flagged = y_pred_t.sum()
    pct_flagged = y_pred_t.mean() * 100
    print(
        f"{thresh:<12.2f} {r:<12.1%} {p:<12.1%} {f1:<12.3f} {n_flagged} ({pct_flagged:.1f}%)"
    )

print("=" * 70)
print(
    f"\nVal Thresh set: {len(y_val_thresh)} samples, {y_val_thresh.sum()} diabetes cases ({y_val_thresh.mean():.1%} prevalence)"
)

# Post Hoc - Threshold optimization

In [ ]:
# Final test evaluation using selected inference model
y_test_proba = inference_model.predict_proba(X_test)[:, 1]
y_pred_final = (y_test_proba >= optimal_thresh).astype(int)

In [ ]:
def bootstrap_metric(y_true, y_pred_or_proba, metric_fn, n_boot=2000, ci=0.95, random_state=RANDOM_STATE):
    """Compute metric with bootstrap confidence interval."""
    rng = np.random.RandomState(random_state)
    scores = []
    n = len(y_true)
    for _ in range(n_boot):
        idx = rng.randint(0, n, size=n)
        try:
            scores.append(metric_fn(y_true.iloc[idx] if hasattr(y_true, 'iloc') else y_true[idx],
                                     y_pred_or_proba[idx]))
        except (ValueError, ZeroDivisionError):
            continue
    scores = np.array(scores)
    alpha = (1 - ci) / 2
    lo, hi = np.percentile(scores, [alpha * 100, (1 - alpha) * 100])
    return np.mean(scores), lo, hi


# Compute metrics with CIs
r_mean, r_lo, r_hi = bootstrap_metric(y_test, y_pred_final, lambda yt, yp: recall_score(yt, yp, zero_division=0))
p_mean, p_lo, p_hi = bootstrap_metric(y_test, y_pred_final, lambda yt, yp: precision_score(yt, yp, zero_division=0))
f1_mean, f1_lo, f1_hi = bootstrap_metric(y_test, y_pred_final, lambda yt, yp: f1_score(yt, yp, zero_division=0))
ap_mean, ap_lo, ap_hi = bootstrap_metric(y_test, y_test_proba, average_precision_score)
brier = brier_score_loss(y_test, y_test_proba)

In [ ]:
print(f"\nFinal chosen threshold: {optimal_thresh:.4f}")
print(f"\n{'Metric':<20} {'Point Est.':<14} {'95% CI':<20}")
print("-" * 54)
print(f"{'Recall':<20} {r_mean:<14.2%} [{r_lo:.2%}, {r_hi:.2%}]")
print(f"{'Precision':<20} {p_mean:<14.2%} [{p_lo:.2%}, {p_hi:.2%}]")
print(f"{'F1 Score':<20} {f1_mean:<14.3f} [{f1_lo:.3f}, {f1_hi:.3f}]")
print(f"{'Average Precision':<20} {ap_mean:<14.3f} [{ap_lo:.3f}, {ap_hi:.3f}]")
print(f"{'Brier Score':<20} {brier:<14.4f}")

In [ ]:
# Confusion matrix at final chosen threshold
cm_final = confusion_matrix(y_test, y_pred_final)  # type: ignore
fig_cm_opt, ax = plt.subplots(figsize=(6, 5))
disp_final = ConfusionMatrixDisplay(
    confusion_matrix=cm_final,
    display_labels=["No Diabetes", "Diabetes"],
)
disp_final.plot(cmap="Blues", ax=ax)  # type: ignore
ax.set_title(
    "Confusion Matrix at Final Threshold\n"
    f"(threshold={optimal_thresh:.3f}, recall={r_mean:.2%}, precision={p_mean:.2%})"
)
plt.tight_layout()
plt.show()  # type: ignore

## 8. Calibration Diagnostics

In [ ]:
# Reliability diagram + Brier score (computed on test set)
y_test_proba_for_cal = inference_model.predict_proba(X_test)[:, 1]

fig_calibration, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: Reliability Diagram ---
ax1 = axes[0]
fraction_of_positives, mean_predicted_value = calibration_curve(
    y_test, y_test_proba_for_cal, n_bins=10, strategy="uniform"
)
ax1.plot(mean_predicted_value, fraction_of_positives, "s-", label="Model", linewidth=2)
ax1.plot([0, 1], [0, 1], "k--", label="Perfectly calibrated")
ax1.set_xlabel("Mean Predicted Probability", fontsize=12)
ax1.set_ylabel("Fraction of Positives", fontsize=12)
ax1.set_title("Reliability Diagram", fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)

# --- Plot 2: Predicted probability distribution ---
ax2 = axes[1]
ax2.hist(y_test_proba_for_cal[y_test == 0], bins=50, alpha=0.6, label="Negative", density=True)
ax2.hist(y_test_proba_for_cal[y_test == 1], bins=50, alpha=0.6, label="Positive", density=True)
ax2.axvline(x=optimal_thresh, color="red", linestyle="--", label=f"Threshold={optimal_thresh:.3f}")
ax2.set_xlabel("Predicted Probability", fontsize=12)
ax2.set_ylabel("Density", fontsize=12)
ax2.set_title("Predicted Probability Distribution by Class", fontsize=14)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
fig_calibration_diag = fig_calibration
plt.show()

brier_test = brier_score_loss(y_test, y_test_proba_for_cal)
print(f"\nBrier Score (test set): {brier_test:.4f}")
print(f"  → 0.0 = perfect calibration, {y_test.mean() * (1 - y_test.mean()):.4f} = baseline (prevalence-based)")

## 9. W&B Final Model Evaluation Logging

This section logs detailed evaluation metrics to Weights & Biases for the winning model.

In [ ]:
# Start a dedicated run for final model analysis
full_params = get_full_params(fixed_params, study.best_params)

final_run = wandb.init(
    project=WANDB_PROJECT,
    entity=ENTITY,
    name=f"{STUDY_NAME}-final-evaluation",
    job_type="evaluation",
    config={
        **full_params,
        "best_cv_ap": study.best_value,
        "chosen_threshold": optimal_thresh,
        "use_calibration": USE_CALIBRATION,
        "calibration_method": "isotonic" if USE_CALIBRATION else None,
    },
)

In [ ]:
# Log interactive PR curve
# W&B expects probabilities for both classes
y_probas_both = inference_model.predict_proba(X_test)

wandb.log({
    "pr_curve": wandb.plot.pr_curve(
        y_true=y_test.values, y_probas=y_probas_both, labels=["No Diabetes", "Diabetes"]
    )
})

In [ ]:
# Log confusion matrix
wandb.log({
    "confusion_matrix": wandb.plot.confusion_matrix(
        y_true=y_test.values,
        preds=y_pred_final,
        class_names=["No Diabetes", "Diabetes"],
    )
})

In [ ]:
# Log threshold analysis table
y_val_proba_final = inference_model.predict_proba(X_val_thresh)[:, 1]

threshold_data = []
for t in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]:
    y_pred_t = (y_val_proba_final >= t).astype(int)
    threshold_data.append([
        t,
        recall_score(y_val_thresh, y_pred_t, zero_division=0),
        precision_score(y_val_thresh, y_pred_t, zero_division=0),
        f1_score(y_val_thresh, y_pred_t, zero_division=0),
        y_pred_t.mean(),
    ])

wandb.log({
    "threshold_analysis": wandb.Table(
        columns=["Threshold", "Recall", "Precision", "F1", "Pct_Flagged"],
        data=threshold_data,
    )
})

In [ ]:
# Log final summary metrics with CIs
y_pred_05 = (y_test_proba >= 0.5).astype(int)

wandb.summary.update({
    # Test set performance at optimal threshold (with CIs)
    "test_recall": r_mean,
    "test_recall_ci_lo": r_lo,
    "test_recall_ci_hi": r_hi,
    "test_precision": p_mean,
    "test_precision_ci_lo": p_lo,
    "test_precision_ci_hi": p_hi,
    "test_f1": f1_mean,
    "test_f1_ci_lo": f1_lo,
    "test_f1_ci_hi": f1_hi,
    "test_ap": ap_mean,
    "test_ap_ci_lo": ap_lo,
    "test_ap_ci_hi": ap_hi,
    "test_brier_score": brier,

    # Test set at default 0.5 threshold
    "test_recall_at_050": recall_score(y_test, y_pred_05),
    "test_precision_at_050": precision_score(y_test, y_pred_05),

    # Model info
    "chosen_threshold": optimal_thresh,
    "best_cv_ap": study.best_value,
    "n_iterations_used": best_model.best_iteration_,
    "use_calibration": USE_CALIBRATION,

    # Data info
    "test_size": len(y_test),
    "val_es_size": len(y_val_es),
    "val_thresh_size": len(y_val_thresh),
    "train_size": len(y_train),
    "test_prevalence": y_test.mean(),
    "n_features": X.shape[1],
    "feature_names": list(X.columns),
})

In [ ]:
# Log SHAP-based feature importance
wandb.log({
    "feature_importance_shap": wandb.Table(
        columns=["Feature", "Mean_Abs_SHAP"],
        data=[[feat, float(imp)] for feat, imp in shap_importance.items()],
    )
})

# Also log as bar chart (top 10)
wandb.log({
    "feature_importance_chart": wandb.plot.bar(
        wandb.Table(
            columns=["Feature", "Importance"],
            data=[[feat, float(imp)] for feat, imp in shap_importance.head(10).items()],
        ),
        "Feature",
        "Importance",
        title="Top 10 Features (mean |SHAP|)",
    )
})

In [ ]:
import os

chart_dir = "wandb_charts"
os.makedirs(chart_dir, exist_ok=True)

charts = {
    "confusion_matrix_050": fig_cm_05,
    "confusion_matrix_optimal": fig_cm_opt,
    "overfitting_curve": fig_overfit,
    "pr_threshold_analysis": fig,
    "shap_global_importance": fig_shap_global,
    "shap_beeswarm": fig_shap_beeswarm,
    "shap_dependence_top3": fig_shap_dep,
    "calibration_diagnostics": fig_calibration_diag,
}

for name, figure in charts.items():
    figure.savefig(f"{chart_dir}/{name}.png", dpi=150, bbox_inches="tight")

artifact = wandb.Artifact(
    name=f"{STUDY_NAME}-charts",
    type="evaluation-charts",
    description="All evaluation charts for final model (includes SHAP and calibration)",
    metadata={
        "threshold_default": 0.5,
        "threshold_optimal": optimal_thresh,
        "test_recall": r_mean,
        "test_recall_95ci": f"[{r_lo:.4f}, {r_hi:.4f}]",
        "test_precision": p_mean,
        "test_brier": brier,
        "best_cv_ap": study.best_value,
    },
)
artifact.add_dir(chart_dir)
wandb.log_artifact(artifact)

In [ ]:
# Log as wandb.Image for quick preview in the run dashboard
wandb.log({name: wandb.Image(figure) for name, figure in charts.items()})

In [ ]:
# Close the W&B run
wandb.finish()

## 10. Model & Test Set Persistence

Saves trained model and test set artifacts for offline FP/FN analysis.

In [ ]:
# =============================================================================
# MODEL & TEST SET PERSISTENCE
# Saves artifacts for offline FP/FN analysis
# =============================================================================

import os
import joblib
from datetime import datetime

# --- Configuration ---
ARTIFACTS_DIR = "./artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

# Versioned filename using study name + timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
artifact_prefix = f"{STUDY_NAME}_{timestamp}"

model_path = f"{ARTIFACTS_DIR}/{artifact_prefix}_model.txt"
test_bundle_path = f"{ARTIFACTS_DIR}/{artifact_prefix}_test_bundle.joblib"
metadata_path = f"{ARTIFACTS_DIR}/{artifact_prefix}_metadata.joblib"

# --- Save LightGBM model (native format) ---
# Note: best_model is the raw LGBMClassifier; inference_model may be calibrated
best_model.booster_.save_model(model_path)

# --- Save test set bundle ---
test_bundle = {
    "X_test": X_test,
    "y_test": y_test,
    "y_test_proba": y_test_proba,
    "y_pred_final": y_pred_final,
    "feature_names": list(X_test.columns),
}
joblib.dump(test_bundle, test_bundle_path)

# --- Save metadata for reproducibility ---
metadata = {
    "study_name": STUDY_NAME,
    "optimal_threshold": optimal_thresh,
    "use_calibration": USE_CALIBRATION,
    "best_params": study.best_params,
    "fixed_params": fixed_params,
    "cat_features": CAT_FEATURES,
    "random_state": RANDOM_STATE,
    "TARGET_COL": TARGET_COL,
    "timestamp": timestamp,
    # Test set metrics for reference
    "test_metrics": {
        "recall": r_mean,
        "recall_ci": (r_lo, r_hi),
        "precision": p_mean,
        "precision_ci": (p_lo, p_hi),
        "f1": f1_mean,
        "f1_ci": (f1_lo, f1_hi),
        "ap": ap_mean,
        "ap_ci": (ap_lo, ap_hi),
        "brier": brier,
    },
}
joblib.dump(metadata, metadata_path)

# --- Confirmation output ---
print("=" * 70)
print("ARTIFACTS SAVED")
print("=" * 70)
print(f"Model:        {model_path}")
print(f"Test bundle:  {test_bundle_path}")
print(f"Metadata:     {metadata_path}")
print(f"\nTest set shape: {X_test.shape}")
print(f"Positive cases in test: {y_test.sum()} ({y_test.mean():.1%})")
print("=" * 70)